# TOP 10 Holdings

INSTRUMENT items in DynamoDB are populated with Top 10 Holdings by the [etrade_morningstar_extract.ipynb](etrade_morningstar_extract.ipynb)

Functions sandboxed in this notebook have been implemented in [DbQuery]()
-   `get_instrument_holdings_df(self, symbols=[])` -- Get the 'top_10_holdings' for a list of instrument symbols
-   `get_position_holdings(self, df_holdings, df_positions)` -- join holdings and accounts where etf_symbol matches the symbol in accounts
-   `get_eft_holdings_by_secname(self, df_position_holdings)` -- holdings grouped by security name and symbol with lists of associated accounts and ETFs

These functions provide detailed and summary information on the 10 top securities held within EFTs to provide a view of the relative proportion of holding within and across portfolios.





In [ ]:
from datetime import datetime
today_str = datetime.now().strftime("%Y-%m-%d")
win_outpath = "/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/excel_pipeline/current"
local_outpath = win_outpath
local_outpath = "./data"

datapath = "/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data"
excel_path = f"{datapath}/excel_pipeline/current"

print(f"Today's date prefix is: {today_str}")
print(f"Windows output path is: {win_outpath}")
print(f"Local output path is: {local_outpath}")
print(f"Excel path is: {excel_path}")

%reload_ext autoreload
%autoreload 2

from ramlytics_database import DynamoDBClient
from ramlytics_database import DbQuery
from ramlytics_yfinance import yf_search

qry = DbQuery(DynamoDBClient())


In [ ]:
# get_intrument_holdings (local copy moved to db_query.py 2026-11-11)
self = qry
from decimal import Decimal

import pandas as pd
from ramlytics_yfinance import yf_search
import logging
logging.basicConfig(level=logging.INFO)

local_outpath = "./data"

def get_instrument_holdings_df(self, symbols=[]):
    if not symbols:
        symbols = self.instruments_df['symbol'].tolist()

    etf_holdings_df = pd.DataFrame()
    for symbol in symbols:
        #print(symbol)
        instrument = self.db.get_instrument(symbol)
        if 'top_10_holdings' not in instrument: # create a 100% placeholder for top 10 holdings             
            instrument['top_10_holdings'] = [{
                "symbol": symbol,
                "name": instrument.get('name', 'n/a'),
                "weight_percent": 100,
                "sector": instrument.get('sector', 'n/a'),
                "industry": instrument.get('industry', 'n/a'),
                "current_price": instrument.get('current_price', 0)
            }]
        # else: 'top_10_holdings' already exists from PDF/JSON import {name: ..., weight_percent: ...}

        holdings = instrument.pop('top_10_holdings')

        pct_in_top10 = 0
        for holding in holdings:
            eft_holding = { "etf_symbol": symbol, **holding}
            pct_in_top10 += holding.get("weight_percent", 0)

            if "symbol" in holding.keys():
                search_result = yf_search(holding['symbol'])
            else:
                search_result = yf_search(holding['name'])

            if search_result:
                logging.info(search_result)
                eft_holding['symbol'] = search_result['symbol']                        
                eft_holding['sector'] = search_result.get('sector','n/a')
                eft_holding['industry'] = search_result.get('industry','n/a')
                eft_holding['current_price'] = round(Decimal(search_result.get('fastinfo',{}).get('lastPrice','0')), 2)
                eft_holding['quote_dt'] = search_result.get('fastinfo',{}).get('quote_dt','n/a')
            else:
                eft_holding = {
                    "etf_symbol": symbol,
                    "name": holding.get('name', 'n/a'),
                    "weight_percent": round(Decimal(holding.get('weight_percent', 0)), 2),
                    "symbol": holding.get('symbol', 'n/a'),
                    "sector": holding.get('sector', 'n/a'),
                    "industry": holding.get('industry', 'n/a'),
                    "current_price":  round(Decimal(instrument.get('fastinfo',{}).get('lastPrice',0.01)), 2)
                }
            
            etf_holdings_df = pd.concat([etf_holdings_df, pd.DataFrame([eft_holding])], ignore_index=True)

        logging.info(f"Symbol: {symbol}, Pct in Top 10: {pct_in_top10}")
        outside_top_10_weight = 100 - pct_in_top10
        if outside_top_10_weight > 0:
            outside_top_10 = {
                "etf_symbol": symbol,
                "symbol": symbol,
                "name": "holdings in addition to top 10",
                "weight_percent": outside_top_10_weight,
                "current_price": instrument.get("current_price", None),
                "quoteType": "EFT",
                "industry": '--',
                "sector": '--',
            }
            etf_holdings_df = pd.concat([etf_holdings_df, pd.DataFrame([outside_top_10])], ignore_index=True)

        etf_holdings_df = etf_holdings_df[['etf_symbol', 'name', 'weight_percent', 'symbol', 'sector', 'industry', 'current_price', 'quote_dt']]
        etf_holdings_df['eft_price_pct'] = etf_holdings_df['weight_percent'] * etf_holdings_df['current_price'] / 100
        logging.info(f"Processed {symbol}")

    return etf_holdings_df

df = get_instrument_holdings_df(self)
now_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
instrument_holdings_file = f"{local_outpath}/instrument_holdings_{now_str}.csv"
df.to_csv(instrument_holdings_file, index=False)
df.head()




In [ ]:
df.head(5)

In [ ]:
# def get_position_holdings(df_holdings, df_positions) - (local copy moved to db_query.py 2026-11-11)

#df_holdings = pd.read_csv(f"{excel_path}/instrument_holdings.csv")
#df_holdings = pd.read_csv(instrument_holdings_file)
df_holdings = df
df_positions = pd.read_csv(f"{excel_path}/account_positions.csv")
now_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

def get_position_holdings(df_holdings, df_positions):
    # join holdings and accounts where etf_symbol matches the symbol in accounts
    df_position_holdings = pd.merge(df_holdings, df_positions, left_on="etf_symbol", right_on="symbol", how="inner")
    df_position_holdings = df_position_holdings[['account_id', 'etf_symbol', 'name', 'weight_percent', 'symbol_x', 'sector', 'industry', 'quote_dt', 'total_value' ]]
    # ToDo:df_position_holdings[['account_id', 'etf_symbol', 'name', 'symbol_x', 'sector', 'industry', 'quote_dt', 'price', 'shares', 'total_value', 'weight_percent', 'holding_value']]
    df_position_holdings = df_position_holdings.rename(columns={"symbol_x": "symbol"})

    df_position_holdings['holding_value'] = round(df_position_holdings['total_value'].astype(float) * (df_position_holdings['weight_percent'] / 100).astype(float), 2)
    df_position_holdings['name'] = df_position_holdings['name'].str.upper()
    df_position_holdings['name'] = df_position_holdings['name'].str.replace(r'[^A-Z0-9 ]', '', regex=True)
    df_position_holdings.fillna('n/a', inplace=True)

    return df_position_holdings

df_position_holdings = get_position_holdings(df_holdings, df_positions)
now_str = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
df_position_holdings.to_csv(f'{local_outpath}/position_holdings_{now_str}.csv', index=False)
df_position_holdings.head()



In [ ]:
def get_eft_holdings_by_secname(df_position_holdings): # (local copy moved to db_query.py 2026-11-11)

    df_eft_holdings_by_secname = df_position_holdings.groupby(["name", "symbol"]).agg(
        total_holding_value=pd.NamedAgg(column="holding_value", aggfunc="sum"),
        account_list=pd.NamedAgg(column="account_id", aggfunc=lambda x: list(set(x))),
        etf_list=pd.NamedAgg(column="etf_symbol", aggfunc=lambda x: list(set(x)))
    ).reset_index()

    total_portfolio_value = round(df_eft_holdings_by_secname["total_holding_value"].sum(), 2)
    df_eft_holdings_by_secname["total_holding_value"] = round(df_eft_holdings_by_secname["total_holding_value"].astype(float), 2)
    df_eft_holdings_by_secname["weight_percent"] = round(df_eft_holdings_by_secname["total_holding_value"] / total_portfolio_value * 100, 2)

    return df_eft_holdings_by_secname

df_eft_holdings_by_secname = get_eft_holdings_by_secname(df_position_holdings)
# convert all total_holding_value to float

df_eft_holdings_by_secname.to_csv(f'{local_outpath}/eft_holdings_by_secname_{now_str}.csv', index=False)
df_eft_holdings_by_secname.head(5)

